# DiatomDINO — public pet-project quickstart

Последовательный цикл без NII и без open-set/Unknown. Долгие этапы запускаются только после проверки флагов `RUN_*`.

In [ ]:
from core.notebook_runtime import bootstrap_notebook, describe_runtime, gpu_preflight, run_guarded
GPU_INDEX = 0
context = bootstrap_notebook(gpu_index=GPU_INDEX)
PROJECT_ROOT = context.project_root
describe_runtime(context)
gpu_preflight(context, minimum_vram_gb=8.0)
RUN_DRY_RUN = False
RUN_DOWNLOAD_AND_BUILD = False
RUN_YOLO = False
RUN_DINO = False
RUN_GALLERY_AND_BENCHMARK = False
RUN_E2E = False


## 1. Данные
Сначала dry-run, затем атомарная загрузка и сборка в `data/`.

In [ ]:
run_guarded(context, context.module_command('scripts.prepare_data', 'all', '--config', 'configs/data.yaml', '--dry-run'), enabled=RUN_DRY_RUN, label='data-dry-run')
run_guarded(context, context.module_command('scripts.prepare_data', 'all', '--config', 'configs/data.yaml'), enabled=RUN_DOWNLOAD_AND_BUILD, label='data-build')


## 2. YOLO на Gunduz

In [ ]:
run_guarded(context, context.module_command('scripts.run_train_detector', '--config', 'configs/detector.yaml'), enabled=RUN_YOLO, label='yolo-train')


## 3. DINOv2 на UDE + Diatom1042 + Siyue Pu

In [ ]:
run_guarded(context, context.module_command('scripts.run_train_classifier', '--config', 'configs/classifier.yaml'), enabled=RUN_DINO, label='dino-train')


## 4. Gunduz gallery и unseen-class retrieval

In [ ]:
run_guarded(context, context.module_command('scripts.run_build_gallery', '--config', 'configs/classifier_benchmark.yaml'), enabled=RUN_GALLERY_AND_BENCHMARK, label='gallery-build')
run_guarded(context, context.module_command('scripts.run_test_classifier', '--config', 'configs/classifier_benchmark.yaml'), enabled=RUN_GALLERY_AND_BENCHMARK, label='retrieval-test')


## 5. Финальный E2E benchmark на Gunduz

In [ ]:
run_guarded(context, context.module_command('scripts.run_test_supermodel', '--config', 'configs/inference.yaml'), enabled=RUN_E2E, label='e2e-test')
